<a href="https://colab.research.google.com/github/Muskan2326/DataScience-ML/blob/main/Tickets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
support_requests = [
    "I forgot my password, how to reset it?",
    "I can't log in, as password is incorrect.",
    "How to see leave balance?",
    "My account is locked, what should I do?",
    "I need to request time off."
]

print("Support Requests:")
for i, request in enumerate(support_requests):
    print(f"{i+1}. {request}")

Support Requests:
1. I forgot my password, how to reset it?
2. I can't log in, as password is incorrect.
3. How to see leave balance?
4. My account is locked, what should I do?
5. I need to request time off.


In [ ]:
from transformers import pipeline

class TicketClassifier:
    def __init__(self, model_name="facebook/bart-large-mnli", confidence_threshold=0.6):
        self.confidence_threshold = confidence_threshold
        try:
            self.classifier = pipeline("zero-shot-classification", model=model_name)
            print(f"Successfully loaded model: {model_name}")
        except Exception as e:
            print(f"Error loading model {model_name}: {e}")
            self.classifier = None

    def classify_tickets(self, tickets, labels):
        if not self.classifier:
            return [{"request": ticket, "category": "Error: Model not loaded", "confidence": 0.0} for ticket in tickets]

        classified_results = []
        predictions = self.classifier(tickets, labels, multi_label=False)

        for i, pred in enumerate(predictions):
            original_request = tickets[i]
            predicted_label = pred['labels'][0]
            confidence_score = pred['scores'][0]

            final_category = "Uncategorized/Manual Review"
            if confidence_score >= self.confidence_threshold:
                if predicted_label == "Authentication":
                    final_category = "Authentication (Password Issues)"
                elif predicted_label == "HR/Administrative":
                    final_category = "HR/Administrative"
                # Add other specific mappings here if needed

            classified_results.append({
                "Original Ticket": original_request,
                "Predicted Category": final_category,
                "Confidence Score": confidence_score
            })
        return classified_results


# Define the labels for zero-shot classification
classification_labels = ["Authentication", "HR/Administrative"]

# Instantiate the classifier
ticket_classifier = TicketClassifier()

# Classify the support requests
if ticket_classifier.classifier:
    results = ticket_classifier.classify_tickets(support_requests, classification_labels)

    # Print the results in a clean table format
    print("\nTicket Classification Results:")
    print("-----------------------------------------------------------------------------------------------------------------------")
    print(f"{'Original Ticket':<50} | {'Predicted Category':<30} | {'Confidence Score':<15}")
    print("-----------------------------------------------------------------------------------------------------------------------")
    for item in results:
        print(f"{item['Original Ticket']:<50} | {item['Predicted Category']:<30} | {item['Confidence Score']:.4f}")
    print("-----------------------------------------------------------------------------------------------------------------------")
else:
    print("Classification cannot proceed due to model loading error.")

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Successfully loaded model: facebook/bart-large-mnli

Ticket Classification Results:
-----------------------------------------------------------------------------------------------------------------------
Original Ticket                                    | Predicted Category             | Confidence Score
-----------------------------------------------------------------------------------------------------------------------
I forgot my password, how to reset it?             | Authentication (Password Issues) | 0.7409
I can't log in, as password is incorrect.          | Authentication (Password Issues) | 0.7881
How to see leave balance?                          | Uncategorized/Manual Review    | 0.5054
My account is locked, what should I do?            | Authentication (Password Issues) | 0.7489
I need to request time off.                        | Uncategorized/Manual Review    | 0.5141
------------------------------------------------------------------------------------------------------

In [ ]:
def classify_ticket(request):
    request_lower = request.lower()
    if "password" in request_lower or "log in" in request_lower or "account locked" in request_lower:
        return "Account/Login Issues"
    elif "leave balance" in request_lower or "time off" in request_lower:
        return "HR/Leave Requests"
    else:
        return "General Inquiry"

# Apply the classification function to each request
classified_tickets = []
for request in support_requests:
    category = classify_ticket(request)
    classified_tickets.append({"request": request, "category": category})

# Display the results
print("\nClassified Tickets:")
for item in classified_tickets:
    print(f"Request: '{item['request']}' -> Category: {item['category']}")


Classified Tickets:
Request: 'I forgot my password, how to reset it?' -> Category: Account/Login Issues
Request: 'I can't log in, as password is incorrect.' -> Category: Account/Login Issues
Request: 'How to see leave balance?' -> Category: HR/Leave Requests
Request: 'My account is locked, what should I do?' -> Category: General Inquiry
Request: 'I need to request time off.' -> Category: HR/Leave Requests
